# Case File: The Lonely Validation Fold

*not looking up from the terminal, not smiling either:*

So. You made it into this one.

Good. This is a `cross_val` script, which means somewhere in here is a partition that isn't a partition, wearing a partition's clothes. It runs. It prints numbers. The numbers even look like the kind of numbers a person would be relieved to see. That's the whole trap, nothing here is going to throw an exception at you and call it a day. You have to go find the crime yourself, same as everyone who's sat in this chair around you.

I'm not hiding a comment that says "bug here" for you. I'm not going to. What I *am* going to give you is the full, correct math, no rationing, because I want you to actually understand cross-validation, not just memorize which line to change. Everyone who skips the math and goes straight for the code ends up "fixing" the wrong thing and feeling very good about it for about four minutes.

Read *carefully*

## Run It

Drop `cs-training.csv` into `./dataset/`, then run `case.py` (or execute the cells below top to bottom) and watch the coverage numbers print.

## Step 0 — Setup

In [1]:
"""
EXPECTED SYMPTOM WHEN RUN:
Fold validation coverage is not a clean partition of the dataset. When each
row is counted for how many folds it appears in as a VALIDATION row, the
counts are not uniformly 1. A large share of rows never appear in any
validation fold, while a comparable number appear in two or more folds. The
reported out-of-fold AUC is computed only over the rows that happened to get
covered at least once, so the script runs cleanly end to end and prints a
plausible-looking number. It does not crash and does not obviously fail.
"""
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import StandardScaler

RANDOM_STATE = 42
N_SPLITS = 5
DATA_PATH = "./dataset/cs-training.csv"

Loading, cleaning, feature prep. Boring on purpose. If the bug were hiding here, this would be a much shorter document.

In [2]:
def load_data(path=DATA_PATH):
    """Load the raw Give Me Some Credit training file."""
    df = pd.read_csv(path, index_col=0)
    return df

In [3]:
def clean_data(df):
    """Impute missing values and drop degenerate rows."""
    df = df.copy()
    df["MonthlyIncome"] = df["MonthlyIncome"].fillna(df["MonthlyIncome"].median())
    df["NumberOfDependents"] = df["NumberOfDependents"].fillna(0)
    df = df[df["age"] > 0]  # remove the known age=0 data entry error
    return df

In [4]:
def prepare_features(df):
    """Split into feature matrix / target vector and scale features."""
    y = df["SeriousDlqin2yrs"].values
    X = df.drop(columns=["SeriousDlqin2yrs"]).values
    X = StandardScaler().fit_transform(X)
    return X, y

## The Math Clue

Sit down. This part I actually care about.

Cross-validation exists to answer one question honestly: *how will this model perform on data it hasn't seen?* You can't just check accuracy on your training set, because a model gets credit for memorizing, not for learning. So you hold some data back. K-fold cross-validation is the disciplined way of doing that holding-back so you don't waste any of your data and don't fool yourself in the process.

Here's the actual definition, formally, because "split it into K pieces" is doing a lot of unexamined work in that sentence. Given a dataset $D = \{1, 2, \dots, n\}$ (just the row indices), K-fold cross-validation requires you to construct subsets $F_1, F_2, \dots, F_K$ such that:

1. **Pairwise disjoint:** $F_i \cap F_j = \emptyset$ for every $i \neq j$. No row is a member of two folds.
2. **Exhaustive:** $\bigcup_{k=1}^{K} F_k = D$. Every row is a member of *some* fold.

Together, those two properties mean $\{F_1, \dots, F_K\}$ is a **partition** of $D$. Not "roughly divided." Not "sampled into K groups." A partition, in the strict set-theoretic sense: disjoint and exhaustive, together.

For each fold $k$, you train a model $f^{(-k)}$ on $D \setminus F_k$ - everything except that fold - and you validate it on $F_k$. Because $f^{(-k)}$ never saw $F_k$ during training, every prediction it makes on $F_k$ is an honest out-of-sample prediction. Stitch those predictions together across all $K$ folds and you get what's called the **out-of-fold (OOF) prediction** for every single row in $D$ — one prediction per row, made by a model that never trained on it. The cross-validated estimate of your loss is:

$$
\widehat{\text{CV}} = \frac{1}{n} \sum_{i=1}^{n} L\big(y_i,\ f^{(-k(i))}(x_i)\big)
$$

where $k(i)$ is "whichever fold contains row $i$." Notice what that notation is quietly assuming: that $k(i)$ is *well-defined* - that there is exactly one fold containing row $i$. That assumption is the partition property, smuggled into the formula as a subscript.

## Your Mission

Somewhere between "load the data" and "print the AUC," the five sets this script calls folds fail to actually partition anything. Go find where the disjoint-and-exhaustive guarantee quietly stopped being a guarantee. Fix it so every row gets exactly one out-of-fold prediction — no more, no less — then rerun the coverage check and watch the holes close.

## Step 1 : Build the folds

In [5]:
def make_cv_folds(n_samples, n_splits=N_SPLITS, random_state=RANDOM_STATE):
    """Build validation index sets for k-fold cross-validation."""
    rng = np.random.RandomState(random_state)
    indices = rng.permutation(n_samples)
    folds = np.array_split(indices, n_splits)
    return folds

## Step 2 : Sanity-check the folds

This is the part where the script tells on itself, if you bother to ask it anything.

In [6]:
def check_fold_coverage(folds, n_samples):
    """Report how many times each row is selected for validation across folds."""
    counts = np.zeros(n_samples, dtype=int)
    for val_idx in folds:
        counts[val_idx] += 1
    print(f"Rows never selected for validation: {(counts == 0).sum()}")
    print(f"Rows selected more than once:       {(counts >= 2).sum()}")
    print(f"Max times any single row was validated on: {counts.max()}")
    return counts

## Step 3 : Train per fold and collect out-of-fold predictions

In [7]:
def run_cross_validation(X, y, folds):
    """Train one model per fold and collect out-of-fold predictions."""
    n = len(y)
    oof_preds = np.full(n, -1.0)
    fold_aucs = []
    for i, val_idx in enumerate(folds):
        train_mask = np.ones(n, dtype=bool)
        train_mask[val_idx] = False
        X_train, y_train = X[train_mask], y[train_mask]
        X_val, y_val = X[val_idx], y[val_idx]
        model = LogisticRegression(max_iter=1000)
        model.fit(X_train, y_train)
        val_probs = model.predict_proba(X_val)[:, 1]
        oof_preds[val_idx] = val_probs
        fold_auc = roc_auc_score(y_val, val_probs)
        fold_aucs.append(fold_auc)
        print(f"Fold {i + 1} AUC: {fold_auc:.4f}")
    return oof_preds, fold_aucs

## Step 4 : Aggregate the out-of-fold score

In [8]:
def evaluate_oof(y, oof_preds):
    """Summarize the pooled out-of-fold AUC over covered rows."""
    covered = oof_preds != -1.0
    print(f"Rows with an out-of-fold prediction: {covered.sum()} / {len(y)}")
    overall_auc = roc_auc_score(y[covered], oof_preds[covered])
    print(f"Overall OOF AUC: {overall_auc:.4f}")
    return overall_auc

## Run the whole thing

In [9]:
df = load_data()
df = clean_data(df)
X, y = prepare_features(df)

folds = make_cv_folds(len(y))
counts = check_fold_coverage(folds, len(y))

oof_preds, fold_aucs = run_cross_validation(X, y, folds)
evaluate_oof(y, oof_preds)

Rows never selected for validation: 49138
Rows selected more than once:       39367
Max times any single row was validated on: 5
Fold 1 AUC: 0.6906
Fold 2 AUC: 0.6988
Fold 3 AUC: 0.6982
Fold 4 AUC: 0.6958
Fold 5 AUC: 0.6963
Rows with an out-of-fold prediction: 100861 / 149999
Overall OOF AUC: 0.6960


0.695957682660835

## Submission

1. Fork this repo.
2. Fix the bug - one root cause, not a rewrite of the whole file.
3. Open a PR against `main` with a short note on what was wrong and why your fix addresses it.
4. A human reviews it and confirms the symptom is actually gone, not just moved somewhere else.
5. On confirmed fix, you get the title below. Yes, it's silly. No, you don't get to skip it.

---
<details>
<summary>Do not open until your PR is confirmed. I mean it.</summary>

## MOCK CEREMONY

*clears throat, entirely too formally for a terminal window*

By the power vested in me by absolutely no one, and in recognition of your discovery that a "no-repeats" sampler run five separate times does not, in fact, add up to five non-repeating sets — I hereby name you:

**Grand Sovereign Partitioner of the Realm, First Warden of the One True Fold, Enemy of the Off-By-Zero-Point-Three-Two-Eight**

Every row gets counted once. Every row gets counted. That's the job. You did the job. Don't let it go to your head. You still write garbage code sometimes, same as the rest of us. Just not today.

</details>